Notebook-style lesson: build a Python environment report.

### Why this matters
When imports fail, packages are missing, or a script behaves differently on one
machine than on another, the most useful question is: which interpreter and
which environment is Python actually using?

This tool answers that in one shot: interpreter version, virtual environment,
import path, and any shadowing of standard library modules.

### Task
Write a script that prints a readable environment summary, detects whether you
are in a virtual environment, reports the import path, and flags shadowing.


In [ ]:


from __future__ import annotations

import json
import site
import sys
from importlib import metadata
from pathlib import Path


def interpreter_info() -> dict[str, str]:
    version = sys.version_info
    return {
        "implementation": sys.implementation.name,
        "version": f"{version.major}.{version.minor}.{version.micro}",
        "executable": sys.executable,
        "prefix": sys.prefix,
        "base_prefix": sys.base_prefix,
    }


def in_virtualenv() -> bool:
    # The interpreter's prefix is the reliable source of truth. `VIRTUAL_ENV`
    # may be stale or absent when the interpreter is launched directly.
    return sys.prefix != sys.base_prefix


def site_packages_dirs() -> list[str]:
    try:
        return list(site.getsitepackages())
    except AttributeError:
        return []


def installed_count() -> int:
    return sum(1 for _ in metadata.distributions())


def check_shadowing(directory: Path | None = None) -> list[tuple[str, str]]:
    directory = directory or Path.cwd()
    clashes: list[tuple[str, str]] = []

    for py in sorted(directory.glob("*.py")):
        if py.stem in sys.stdlib_module_names:
            clashes.append((py.name, py.stem))

    for pkg in sorted(p for p in directory.iterdir() if p.is_dir()):
        if (pkg / "__init__.py").exists() and pkg.name in sys.stdlib_module_names:
            clashes.append((f"{pkg.name}/", pkg.name))

    return clashes


def render_text() -> int:
    info = interpreter_info()

    print("\nInterpreter")
    print(f"  implementation : {info['implementation']} {info['version']}")
    print(f"  executable     : {info['executable']}")
    print(f"  prefix         : {info['prefix']}")
    print(f"  base_prefix    : {info['base_prefix']}")

    print("\nEnvironment")
    print(f"  virtualenv     : {'YES' if in_virtualenv() else 'NO'}")
    print(f"  site-packages  : {site_packages_dirs()[:1] or 'n/a'}")
    print(f"  installed      : {installed_count()} distributions")

    print("\nImport path (sys.path)")
    for index, path in enumerate(sys.path):
        marker = " <- searched FIRST" if index == 0 else ""
        print(f"  [{index}] {path}{marker}")

    print("\nShadowing check")
    clashes = check_shadowing()
    if not clashes:
        print("  OK   no local files shadow a standard library module")
        return 0

    for filename, module in clashes:
        print(f"  WARN {filename} shadows '{module}'")
    return 1


def render_json() -> int:
    payload = {
        "interpreter": interpreter_info(),
        "virtualenv": in_virtualenv(),
        "site_packages": site_packages_dirs(),
        "installed_distributions": installed_count(),
        "sys_path": sys.path,
        "shadowing": [{"file": file_name, "module": module} for file_name, module in check_shadowing()],
    }
    print(json.dumps(payload, indent=2))
    return 1 if check_shadowing() else 0


def main(argv: list[str]) -> int:
    if "--json" in argv:
        return render_json()
    return render_text()


if __name__ == "__main__":
    raise SystemExit(main(sys.argv[1:]))
